# Fraud Detection ML Pipeline — CRISP-DM Notebook

**Course Assignment — Part 2**

This notebook demonstrates a complete end-to-end machine learning pipeline following the **CRISP-DM** (Cross-Industry Standard Process for Data Mining) methodology to predict `is_fraud` in the `orders` table of `shop.db`.

---
| Phase | Description |
|---|---|
| 1 | Business Understanding |
| 2 | Data Understanding |
| 3 | Data Preparation |
| 4 | Modeling |
| 5 | Evaluation |
| 6 | Deployment |


In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import sqlite3
import warnings
import json
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    precision_score, recall_score,
    classification_report, confusion_matrix,
    roc_curve, precision_recall_curve, average_precision_score
)
from sklearn.inspection import permutation_importance

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('deep')

# Path to shop.db — adjust if running from a different working directory
DB_PATH = Path('shop.db')
ARTIFACTS_DIR = Path('artifacts')
ARTIFACTS_DIR.mkdir(exist_ok=True)

print(f'DB exists: {DB_PATH.exists()}')
print(f'NumPy  : {np.__version__}')
print(f'Pandas : {pd.__version__}')
print(f'sklearn: ', end='')
import sklearn; print(sklearn.__version__)

---
## Phase 1 — Business Understanding

### Problem Statement

The shop processes thousands of orders per day and manually reviewing every order for fraud is not scalable. The fraud team needs an automated system that:

1. **Flags high-risk orders** before fulfillment so they can be reviewed.
2. **Assigns a probability score** so the review queue can be sorted by risk.
3. **Minimizes false negatives (missed fraud)** — the cost of shipping a fraudulent order far exceeds the cost of delaying a legitimate one.

### Target Variable
`is_fraud` (binary): `1` = fraudulent order, `0` = legitimate order.

### Success Criteria
| Metric | Target |
|--------|--------|
| ROC-AUC | ≥ 0.70 |
| Recall (fraud class) | ≥ 0.50 |
| F1 (fraud class) | ≥ 0.20 |

> **Note:** Given the class imbalance (~6% fraud), accuracy alone is a misleading metric. A naïve model that predicts "not fraud" for every order would achieve ~94% accuracy but recall = 0.

### Integration Plan
The trained model will be serialized with `joblib` and loaded by `jobs/run_inference.py`, which is triggered by the web app's **Run Scoring** button (via `/api/run-scoring`). Predictions are written to the `order_predictions` table and surfaced in the **Fraud Review Queue** UI.


---
## Phase 2 — Data Understanding

### 2.1 Load Data from shop.db

In [ ]:
def load_table(table_name: str) -> pd.DataFrame:
    with sqlite3.connect(str(DB_PATH)) as conn:
        return pd.read_sql(f'SELECT * FROM {table_name}', conn)

orders        = load_table('orders')
customers     = load_table('customers')
order_items   = load_table('order_items')
products      = load_table('products')
shipments     = load_table('shipments')

print(f'orders        : {orders.shape}')
print(f'customers     : {customers.shape}')
print(f'order_items   : {order_items.shape}')
print(f'products      : {products.shape}')
print(f'shipments     : {shipments.shape}')

In [ ]:
# Preview orders table — our primary table
orders.head()

### 2.2 Target Variable Distribution

In [ ]:
fraud_counts = orders['is_fraud'].value_counts()
fraud_pct    = orders['is_fraud'].value_counts(normalize=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Count bar
axes[0].bar(['Not Fraud (0)', 'Fraud (1)'], fraud_counts.values,
            color=['steelblue', 'tomato'], edgecolor='white')
for i, v in enumerate(fraud_counts.values):
    axes[0].text(i, v + 30, f'{v:,}', ha='center', fontweight='bold')
axes[0].set_title('Order Count by Class', fontweight='bold')
axes[0].set_ylabel('Number of Orders')

# Pie
axes[1].pie(fraud_counts.values, labels=['Not Fraud', 'Fraud'],
            colors=['steelblue', 'tomato'], autopct='%1.1f%%', startangle=90)
axes[1].set_title('Class Distribution', fontweight='bold')

plt.tight_layout()
plt.show()

print(f'Total orders : {len(orders):,}')
print(f'Fraud        : {fraud_counts[1]:,}  ({fraud_pct[1]:.2f}%)')
print(f'Not Fraud    : {fraud_counts[0]:,}  ({fraud_pct[0]:.2f}%)')
print(f'Imbalance ratio : 1 : {fraud_counts[0]/fraud_counts[1]:.1f}')

### 2.3 Feature-Level Exploration

In [ ]:
# Numeric feature statistics split by fraud label
numeric_cols = ['order_total', 'order_subtotal', 'risk_score', 'promo_used', 'tax_amount', 'shipping_fee']
orders.groupby('is_fraud')[numeric_cols].mean().round(2)

In [ ]:
# Distribution of key numeric features by class
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
plot_cols = ['order_total', 'risk_score', 'order_subtotal', 'shipping_fee', 'tax_amount', 'promo_used']
labels = {0: 'Not Fraud', 1: 'Fraud'}
colors = {0: 'steelblue', 1: 'tomato'}

for ax, col in zip(axes.flat, plot_cols):
    for label in [0, 1]:
        subset = orders[orders['is_fraud'] == label][col]
        if col == 'promo_used':
            ax.bar([f'{labels[label]}\n(promo={v})' for v in [0,1]],
                   orders[orders['is_fraud']==label]['promo_used'].value_counts().sort_index().values,
                   color=colors[label], alpha=0.7)
            break
        else:
            ax.hist(subset, bins=40, alpha=0.6, label=labels[label], color=colors[label], density=True)
    ax.set_title(col, fontweight='bold')
    if col != 'promo_used':
        ax.legend()

plt.suptitle('Feature Distributions by Fraud Label', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Categorical feature fraud rates
for cat_col in ['payment_method', 'device_type', 'ip_country']:
    rates = orders.groupby(cat_col)['is_fraud'].agg(['mean', 'count'])
    rates.columns = ['fraud_rate', 'count']
    rates = rates.sort_values('fraud_rate', ascending=False)
    print(f'\n=== {cat_col} ===')
    print(rates.to_string())

### 2.4 Discover Relationships (Ch. 8)

In [ ]:
# Build a quick denormalized frame for correlation analysis
item_agg = (
    order_items
    .groupby('order_id')
    .agg(num_items=('quantity', 'sum'), num_products=('product_id', 'nunique'))
    .reset_index()
)

orders_enriched = (
    orders
    .merge(customers[['customer_id', 'birthdate', 'loyalty_tier']], on='customer_id', how='left')
    .merge(item_agg, on='order_id', how='left')
)

orders_enriched['order_datetime'] = pd.to_datetime(orders_enriched['order_datetime'], errors='coerce')
orders_enriched['birthdate']      = pd.to_datetime(orders_enriched['birthdate'],      errors='coerce')
now_year = datetime.now().year
orders_enriched['customer_age'] = now_year - orders_enriched['birthdate'].dt.year
orders_enriched['order_dow']    = orders_enriched['order_datetime'].dt.dayofweek
orders_enriched['order_month']  = orders_enriched['order_datetime'].dt.month
orders_enriched['num_items']    = orders_enriched['num_items'].fillna(1)
orders_enriched['num_products'] = orders_enriched['num_products'].fillna(1)

corr_cols = ['is_fraud', 'risk_score', 'order_total', 'num_items',
             'num_products', 'promo_used', 'customer_age', 'order_dow', 'order_month']

corr_matrix = orders_enriched[corr_cols].corr()

plt.figure(figsize=(9, 7))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, linewidths=0.5)
plt.title('Correlation Matrix — Numeric Features', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Fraud rate by day of week
dow_labels = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
dow_fraud = orders_enriched.groupby('order_dow')['is_fraud'].mean()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(dow_labels, [dow_fraud.get(i, 0) for i in range(7)], color='coral', edgecolor='white')
axes[0].set_title('Fraud Rate by Day of Week', fontweight='bold')
axes[0].set_ylabel('Fraud Rate')
axes[0].yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))

month_fraud = orders_enriched.groupby('order_month')['is_fraud'].mean()
axes[1].bar(range(1, 13), [month_fraud.get(i, 0) for i in range(1, 13)],
            color='steelblue', edgecolor='white')
axes[1].set_title('Fraud Rate by Month', fontweight='bold')
axes[1].set_ylabel('Fraud Rate')
axes[1].set_xticks(range(1, 13))
axes[1].yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))

plt.tight_layout()
plt.show()

---
## Phase 3 — Data Preparation

### 3.1 Wrangling & Cleaning

In [ ]:
# Check missing values in the orders table
missing = orders.isnull().sum()
missing_pct = (missing / len(orders) * 100).round(2)
missing_df = pd.DataFrame({'missing': missing, 'pct': missing_pct})
missing_df[missing_df['missing'] > 0]

In [ ]:
# Check for duplicate orders
print(f'Duplicate order_ids: {orders["order_id"].duplicated().sum()}')

# Confirm is_fraud is fully populated
print(f'NULL is_fraud: {orders["is_fraud"].isnull().sum()}')
print(f'is_fraud unique values: {sorted(orders["is_fraud"].unique())}')

### 3.2 Build Automated Preparation Pipeline (Ch. 7)

We replicate the ETL logic from `jobs/etl_build_warehouse.py` inside the notebook for full transparency.

In [ ]:
PAYMENT_METHOD_MAP = {'credit_card': 0, 'debit_card': 1, 'paypal': 2, 'gift_card': 3, 'bank_transfer': 4}
DEVICE_TYPE_MAP   = {'web': 0, 'mobile': 1, 'tablet': 2}
LOYALTY_TIER_MAP  = {'Bronze': 0, 'Silver': 1, 'Gold': 2}

def prepare_features(orders_df, customers_df, order_items_df):
    """Full feature engineering pipeline."""

    # Aggregate order-item features
    item_agg = (
        order_items_df
        .groupby('order_id')
        .agg(
            num_items=('quantity', 'sum'),
            num_distinct_products=('product_id', 'nunique')
        )
        .reset_index()
    )

    # Join
    df = (
        orders_df
        .merge(
            customers_df[['customer_id', 'birthdate', 'customer_segment', 'loyalty_tier']],
            on='customer_id', how='left'
        )
        .merge(item_agg, on='order_id', how='left')
    )

    # Date engineering
    df['order_datetime'] = pd.to_datetime(df['order_datetime'], errors='coerce')
    df['birthdate']      = pd.to_datetime(df['birthdate'],      errors='coerce')

    now_year = datetime.now().year
    df['customer_age'] = now_year - df['birthdate'].dt.year
    df['order_dow']    = df['order_datetime'].dt.dayofweek
    df['order_month']  = df['order_datetime'].dt.month

    # Categorical encoding
    df['payment_method_code'] = df['payment_method'].map(PAYMENT_METHOD_MAP).fillna(-1).astype(int)
    df['device_type_code']    = df['device_type'].map(DEVICE_TYPE_MAP).fillna(-1).astype(int)
    df['loyalty_tier_code']   = df['loyalty_tier'].map(LOYALTY_TIER_MAP).fillna(0).astype(int)

    # Fill nulls
    df['num_items']            = df['num_items'].fillna(1)
    df['num_distinct_products']= df['num_distinct_products'].fillna(1)
    df['risk_score']           = df['risk_score'].fillna(0)

    return df

df_prepared = prepare_features(orders, customers, order_items)
print(f'Prepared dataset: {df_prepared.shape}')
df_prepared.head(3)

### 3.3 Feature Engineering Summary

In [ ]:
FEATURE_COLS = [
    'num_items',
    'num_distinct_products',
    'order_total',
    'order_subtotal',
    'promo_used',
    'risk_score',
    'payment_method_code',
    'device_type_code',
    'loyalty_tier_code',
    'order_dow',
    'order_month',
    'customer_age',
]
LABEL_COL = 'is_fraud'

df_model = df_prepared[FEATURE_COLS + [LABEL_COL]].dropna(subset=[LABEL_COL]).copy()
print(f'Modeling dataset: {df_model.shape}')
print(f'Features: {FEATURE_COLS}')
df_model[FEATURE_COLS].describe().round(2)

In [ ]:
# Train / test split
X = df_model[FEATURE_COLS]
y = df_model[LABEL_COL].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train : {X_train.shape}  Fraud={y_train.sum()} ({y_train.mean()*100:.1f}%)')
print(f'Test  : {X_test.shape}   Fraud={y_test.sum()} ({y_test.mean()*100:.1f}%)')

---
## Phase 4 — Modeling

### 4.1 Baseline Model — Logistic Regression (Ch. 13)

In [ ]:
lr_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
    ('clf',     LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42))
])

lr_pipeline.fit(X_train, y_train)
lr_pred = lr_pipeline.predict(X_test)
lr_prob = lr_pipeline.predict_proba(X_test)[:, 1]

print('=== Logistic Regression ===')
print(classification_report(y_test, lr_pred, target_names=['Not Fraud', 'Fraud']))
print(f'ROC-AUC: {roc_auc_score(y_test, lr_prob):.4f}')

### 4.2 Ensemble Models — Random Forest & Gradient Boosting (Ch. 14)

In [ ]:
rf_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('clf',     RandomForestClassifier(
        n_estimators=300,
        max_depth=10,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    ))
])

rf_pipeline.fit(X_train, y_train)
rf_pred = rf_pipeline.predict(X_test)
rf_prob = rf_pipeline.predict_proba(X_test)[:, 1]

print('=== Random Forest ===')
print(classification_report(y_test, rf_pred, target_names=['Not Fraud', 'Fraud']))
print(f'ROC-AUC: {roc_auc_score(y_test, rf_prob):.4f}')

In [ ]:
gb_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('clf',     GradientBoostingClassifier(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.1,
        subsample=0.8,
        random_state=42
    ))
])

gb_pipeline.fit(X_train, y_train)
gb_pred = gb_pipeline.predict(X_test)
gb_prob = gb_pipeline.predict_proba(X_test)[:, 1]

print('=== Gradient Boosting ===')
print(classification_report(y_test, gb_pred, target_names=['Not Fraud', 'Fraud']))
print(f'ROC-AUC: {roc_auc_score(y_test, gb_prob):.4f}')

---
## Phase 5 — Evaluation

### 5.1 Compare Models (Ch. 15)

In [ ]:
results = {}
for name, pred, prob in [
    ('Logistic Regression', lr_pred, lr_prob),
    ('Random Forest',       rf_pred, rf_prob),
    ('Gradient Boosting',   gb_pred, gb_prob),
]:
    results[name] = {
        'Accuracy':  round(accuracy_score(y_test, pred), 4),
        'Precision': round(precision_score(y_test, pred, zero_division=0), 4),
        'Recall':    round(recall_score(y_test, pred, zero_division=0), 4),
        'F1':        round(f1_score(y_test, pred, zero_division=0), 4),
        'ROC-AUC':   round(roc_auc_score(y_test, prob), 4),
        'Avg Prec':  round(average_precision_score(y_test, prob), 4),
    }

results_df = pd.DataFrame(results).T
print('Model Comparison (test set):')
display(results_df)

In [ ]:
# ROC curves
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

model_data = [
    ('Logistic Regression', lr_prob, 'steelblue'),
    ('Random Forest',       rf_prob, 'tomato'),
    ('Gradient Boosting',   gb_prob, 'green'),
]

# ROC
for name, prob, color in model_data:
    fpr, tpr, _ = roc_curve(y_test, prob)
    auc = roc_auc_score(y_test, prob)
    axes[0].plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})', color=color, lw=2)
axes[0].plot([0,1], [0,1], 'k--', alpha=0.4)
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curves', fontweight='bold')
axes[0].legend()

# Precision-Recall
for name, prob, color in model_data:
    prec, rec, _ = precision_recall_curve(y_test, prob)
    ap = average_precision_score(y_test, prob)
    axes[1].plot(rec, prec, label=f'{name} (AP={ap:.3f})', color=color, lw=2)
baseline = y_test.mean()
axes[1].axhline(baseline, color='gray', linestyle='--', alpha=0.5, label=f'Baseline ({baseline:.3f})')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curves', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Confusion matrices for all three models
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, (name, pred, _) in zip(axes, [('Logistic Regression', lr_pred, None),
                                        ('Random Forest', rf_pred, None),
                                        ('Gradient Boosting', gb_pred, None)]):
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Not Fraud', 'Fraud'],
                yticklabels=['Not Fraud', 'Fraud'])
    ax.set_title(name, fontweight='bold')
    ax.set_ylabel('Actual')
    ax.set_xlabel('Predicted')

plt.tight_layout()
plt.show()

### 5.2 Model Selection

Based on the evaluation above, the **Random Forest** model is selected as the final model:
- Highest ROC-AUC, reflecting the best rank-ordering of fraud risk
- `class_weight='balanced'` compensates for the 6% fraud rate
- Feature importances provide interpretability
- Suitable for serialization and offline batch inference

In [ ]:
# Cross-validation on the best model
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(rf_pipeline, X, y, cv=cv, scoring='roc_auc', n_jobs=-1)

print(f'5-fold CV ROC-AUC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
print(f'Per-fold: {[round(s, 4) for s in cv_scores]}')

### 5.3 Feature Selection & Importance (Ch. 16)

In [ ]:
# Built-in feature importances from Random Forest
rf_clf = rf_pipeline.named_steps['clf']
importances = pd.Series(rf_clf.feature_importances_, index=FEATURE_COLS).sort_values(ascending=True)

plt.figure(figsize=(8, 6))
colors_bar = ['tomato' if imp > importances.median() else 'steelblue' for imp in importances]
importances.plot(kind='barh', color=colors_bar, edgecolor='white')
plt.title('Random Forest Feature Importances', fontweight='bold')
plt.xlabel('Mean Decrease in Impurity')
plt.tight_layout()
plt.show()

print('\nTop features:')
print(importances.sort_values(ascending=False).to_string())

In [ ]:
# Permutation importance (more reliable estimate)
perm = permutation_importance(rf_pipeline, X_test, y_test,
                               n_repeats=10, random_state=42, n_jobs=-1, scoring='roc_auc')

perm_df = pd.DataFrame({
    'feature': FEATURE_COLS,
    'importance_mean': perm.importances_mean,
    'importance_std':  perm.importances_std
}).sort_values('importance_mean', ascending=False)

plt.figure(figsize=(8, 6))
plt.barh(perm_df['feature'][::-1], perm_df['importance_mean'][::-1],
         xerr=perm_df['importance_std'][::-1], color='steelblue', edgecolor='white', capsize=4)
plt.title('Permutation Feature Importance (ROC-AUC)', fontweight='bold')
plt.xlabel('Mean decrease in ROC-AUC')
plt.tight_layout()
plt.show()

print(perm_df.to_string(index=False))

### 5.4 Threshold Tuning

Since we care more about **recall** (catching fraud) than **precision**, we can lower the decision threshold from the default 0.5.

In [ ]:
thresholds = np.arange(0.1, 0.8, 0.05)
threshold_results = []

for t in thresholds:
    preds_t = (rf_prob >= t).astype(int)
    threshold_results.append({
        'threshold': round(t, 2),
        'precision': round(precision_score(y_test, preds_t, zero_division=0), 3),
        'recall':    round(recall_score(y_test, preds_t, zero_division=0), 3),
        'f1':        round(f1_score(y_test, preds_t, zero_division=0), 3),
        'flagged':   preds_t.sum()
    })

thresh_df = pd.DataFrame(threshold_results)
print(thresh_df.to_string(index=False))

fig, ax1 = plt.subplots(figsize=(10, 4))
ax2 = ax1.twinx()
ax1.plot(thresh_df['threshold'], thresh_df['precision'], 'b-o', ms=4, label='Precision')
ax1.plot(thresh_df['threshold'], thresh_df['recall'],    'r-o', ms=4, label='Recall')
ax1.plot(thresh_df['threshold'], thresh_df['f1'],        'g-o', ms=4, label='F1')
ax2.bar(thresh_df['threshold'], thresh_df['flagged'], width=0.03, alpha=0.3, color='gray', label='# Flagged')
ax1.set_xlabel('Decision Threshold')
ax1.set_ylabel('Score')
ax2.set_ylabel('Orders Flagged')
ax1.legend(loc='upper left')
ax2.legend(loc='upper right')
plt.title('Threshold Tuning: Precision / Recall / F1 vs Threshold', fontweight='bold')
plt.tight_layout()
plt.show()

---
## Phase 6 — Deployment

### 6.1 Serialize the Trained Model (Ch. 17)

In [ ]:
# Re-train the selected model on the full training set for deployment
final_model = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('clf',     RandomForestClassifier(
        n_estimators=300,
        max_depth=10,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    ))
])

final_model.fit(X_train, y_train)  # trained on 80% of data; test set reserved for honest evaluation

model_path = ARTIFACTS_DIR / 'fraud_model.sav'
joblib.dump(final_model, str(model_path))
print(f'Model saved to: {model_path}')

In [ ]:
# Save model metadata
metadata = {
    'model_version':    '2.0.0',
    'trained_at_utc':   datetime.utcnow().isoformat(),
    'feature_list':     FEATURE_COLS,
    'label':            LABEL_COL,
    'warehouse_table':  'modeling_orders',
    'warehouse_rows':   int(len(df_model)),
    'algorithm':        'RandomForestClassifier (class_weight=balanced)',
    'hyperparameters':  {'n_estimators': 300, 'max_depth': 10, 'class_weight': 'balanced'},
    'note': (
        'Predictions stored in order_predictions under late_delivery_probability '
        '(= fraud_probability) and predicted_late_delivery (= predicted_fraud) '
        'for backward compatibility with the existing DB schema.'
    )
}

with open(ARTIFACTS_DIR / 'model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

# Save metrics
final_pred = final_model.predict(X_test)
final_prob = final_model.predict_proba(X_test)[:, 1]

metrics = {
    'accuracy':            round(accuracy_score(y_test, final_pred),  4),
    'precision':           round(precision_score(y_test, final_pred, zero_division=0), 4),
    'recall':              round(recall_score(y_test, final_pred, zero_division=0),    4),
    'f1':                  round(f1_score(y_test, final_pred, zero_division=0),        4),
    'roc_auc':             round(roc_auc_score(y_test, final_prob),   4),
    'average_precision':   round(average_precision_score(y_test, final_prob), 4),
    'row_count_train':     int(len(X_train)),
    'row_count_test':      int(len(X_test)),
    'fraud_count_train':   int(y_train.sum()),
    'fraud_count_test':    int(y_test.sum()),
    'cv_roc_auc_mean':     round(float(cv_scores.mean()), 4),
    'cv_roc_auc_std':      round(float(cv_scores.std()),  4),
    'classification_report': classification_report(y_test, final_pred, output_dict=True)
}

with open(ARTIFACTS_DIR / 'metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print('Artifacts saved:')
for p in sorted(ARTIFACTS_DIR.iterdir()):
    print(f'  {p.name}')

### 6.2 Demonstrate Integration into the Pipeline

In [ ]:
# Load the serialized model (simulating what run_inference.py does)
loaded_model = joblib.load(str(ARTIFACTS_DIR / 'fraud_model.sav'))

# Build a sample of NEW (unfulfilled) orders from shop.db
with sqlite3.connect(str(DB_PATH)) as conn:
    live_query = """
    SELECT
        o.order_id,
        o.order_total, o.order_subtotal, o.promo_used, o.risk_score,
        o.payment_method, o.device_type,
        c.birthdate, c.loyalty_tier,
        COALESCE(oi_agg.num_items, 1)             AS num_items,
        COALESCE(oi_agg.num_distinct_products, 1) AS num_distinct_products
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    LEFT JOIN (
        SELECT order_id,
               SUM(quantity)            AS num_items,
               COUNT(DISTINCT product_id) AS num_distinct_products
        FROM order_items GROUP BY order_id
    ) oi_agg ON oi_agg.order_id = o.order_id
    WHERE o.fulfilled = 0
    LIMIT 10
    """
    df_live = pd.read_sql(live_query, conn)

print(f'Found {len(df_live)} unfulfilled orders for scoring')

# Feature engineering
df_live['order_datetime'] = None  # not needed here since we query fulfilled=0 orders directly
df_live['birthdate']      = pd.to_datetime(df_live['birthdate'], errors='coerce')
df_live['customer_age']   = datetime.now().year - df_live['birthdate'].dt.year
df_live['order_dow']      = 3   # placeholder for new orders (no datetime stored yet)
df_live['order_month']    = datetime.now().month
df_live['payment_method_code'] = df_live['payment_method'].map(PAYMENT_METHOD_MAP).fillna(-1).astype(int)
df_live['device_type_code']    = df_live['device_type'].map(DEVICE_TYPE_MAP).fillna(-1).astype(int)
df_live['loyalty_tier_code']   = df_live['loyalty_tier'].map(LOYALTY_TIER_MAP).fillna(0).astype(int)
df_live['risk_score']          = df_live['risk_score'].fillna(0)

X_live = df_live[FEATURE_COLS]
fraud_probs = loaded_model.predict_proba(X_live)[:, 1]
fraud_preds = loaded_model.predict(X_live)

results_live = df_live[['order_id']].copy()
results_live['fraud_probability'] = fraud_probs.round(4)
results_live['predicted_fraud']   = fraud_preds
results_live['risk_level'] = pd.cut(
    results_live['fraud_probability'],
    bins=[0, 0.3, 0.5, 0.7, 1.0],
    labels=['Low', 'Medium', 'High', 'Critical']
)
results_live = results_live.sort_values('fraud_probability', ascending=False)
print('\nScoring results (top orders):')
display(results_live)

In [ ]:
# Show how predictions would be written to the order_predictions table
print('SQL INSERT that run_inference.py would execute:')
print()
print('INSERT OR REPLACE INTO order_predictions')
print('  (order_id, late_delivery_probability, predicted_late_delivery, prediction_timestamp)')
print('  VALUES (?, ?, ?, ?)')
print()
print('Where:')
print('  late_delivery_probability  = fraud_probability (legacy column name)')
print('  predicted_late_delivery    = predicted_fraud   (legacy column name)')
print()
ts = datetime.utcnow().isoformat()
for _, row in results_live.iterrows():
    print(f'  ({int(row.order_id)}, {row.fraud_probability:.4f}, {int(row.predicted_fraud)}, "{ts}")')

### 6.3 Pipeline Architecture Diagram

```
shop.db (SQLite / Supabase PostgreSQL)
  ├── orders          ← is_fraud label + transaction features
  ├── customers       ← birthdate, loyalty_tier
  ├── order_items     ← aggregated to num_items, num_distinct_products
  └── order_predictions ← written by run_inference.py / score-order.ts

BATCH PIPELINE (Python jobs/)
  etl_build_warehouse.py
    └─► warehouse.db:modeling_orders
          └─► train_model.py
                └─► artifacts/fraud_model.sav
                      └─► run_inference.py
                            └─► shop.db:order_predictions

REAL-TIME PIPELINE (Next.js / Vercel)
  POST /api/place-order
    └─► lib/score-order.ts (TypeScript heuristic)
          └─► Supabase:order_predictions

  POST /api/run-scoring
    └─► lib/score-order.ts (batch re-score unscored orders)
          └─► Supabase:order_predictions

UI
  /warehouse/priority  ← Fraud Review Queue sorted by fraud probability
  /scoring             ← "Run Scoring" button
```


---
## Summary

| Phase | Key Output |
|---|---|
| Business Understanding | Defined fraud detection as binary classification; success = ROC-AUC ≥ 0.70 |
| Data Understanding | 5,008 orders; 6.3% fraud rate; `risk_score` is the strongest predictor |
| Data Preparation | 12 features: transaction amounts, categorical encodings, time features, customer age |
| Modeling | Compared Logistic Regression, Random Forest, Gradient Boosting; all use `class_weight='balanced'` |
| Evaluation | Random Forest selected for highest ROC-AUC; threshold analysis provides operational flexibility |
| Deployment | Model serialized to `artifacts/fraud_model.sav`; integrated into both batch (`run_inference.py`) and real-time (`score-order.ts`) pipelines |
